# Semantic Search for Lenta.ru News
## Business Task

The user enters a **search query** (“inflation”, “ruble exchange rate”, “Champions League final”), and the system returns relevant articles, even if the exact query words do not appear in the text.

**Goal**: compare a lexical baseline (TF-IDF) with a semantic approach (Word2Vec + cosine similarity) in a production-style setup: configuration, classes, train/index split, metrics, and artifacts.

## Dataset

[zloelias/lenta-ru](https://huggingface.co/datasets/zloelias/lenta-ru) — a Russian news dataset containing categories (topic), headlines, and article texts. It is suitable for search tasks and topic-based validation.

## Pipeline

```text
Loading → EDA → preprocessing → split (train/val/test)
    → indexing (TF-IDF | W2V document vectors) using train only
    → search API → validation (MRR, Recall@k, Topic-Precision@k)
    → method comparison → artifact saving
```

In [26]:
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
import json

import re

import nltk
from nltk.corpus import stopwords as nltk_stopwords
from razdel import tokenize as razdel_tokenize

import gensim
from gensim.models import Word2Vec

from datasets import load_dataset
from dataclasses import asdict, dataclass, field

import sklearn
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)

In [32]:
@dataclass(frozen=True)
class SemanticSearchConfig:
    dataset_id: str = "zloelias/lenta-ru"
    random_state: int = 42
    corpus_sample: int = 8_000

    min_tokens_per_doc: int = 12
    min_word_len: int = 2

    title_boost: int = 3
    tfidf_max_features: int = 20_000
    embed_dim: int = 128

    w2v_window: int = 5
    w2v_min_count: int = 3
    w2v_epochs: int = 15

    lsa_components: int = 200

    search_top_k: int = 10

    hybrid_candidates: int = 50
    hybrid_alpha: float = 0.85

    title_query_tokens: int = 5

    artifacts_dir: Path = field(default_factory=lambda: Path("artifacts"))

    def to_dict(self) -> dict:
        d = asdict(self)
        d["artifacts_dir"] = str(self.artifacts_dir)
        return d

In [31]:
CFG = SemanticSearchConfig()

CFG.artifacts_dir.mkdir(parents=True, exist_ok=True)
np.random.seed(CFG.random_state)